In [7]:
import pandas as pd
import re
import spacy
import gradio as gr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import pickle

nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

In [2]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("Resume.csv")

Saving Resume.csv to Resume (1).csv


In [8]:
df_clean = df[['Resume_str', 'Category']].copy()
df_clean.columns = ['Resume', 'Category']

def clean_resume(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and len(token.text) > 2]
    return " ".join(tokens)

print("Preprocessing 2400+ resumes... This takes about 2-3 minutes with Lemmatization.")
df_clean['Cleaned_Resume'] = df_clean['Resume'].apply(clean_resume)


df_clean.drop_duplicates(subset=['Cleaned_Resume'], inplace=True)
df_clean.reset_index(drop=True, inplace=True)

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X = tfidf.fit_transform(df_clean['Cleaned_Resume'])
y = df_clean['Category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

print(f"Training Complete! Accuracy: {model.score(X_test, y_test):.2%}")

Preprocessing 2400+ resumes... This takes about 2-3 minutes with Lemmatization.
Training Complete! Accuracy: 72.23%


In [12]:
import fitz
import gradio as gr
def predict_resume(text_input, file_input):
    raw_text = ""
    if file_input is not None:
        try:
            with fitz.open(file_input.name) as doc:
                for page in doc:
                    raw_text += page.get_text()
        except Exception as e:
            return {"Error": f"Could not read PDF: {str(e)}"}
    elif text_input and text_input.strip():
        raw_text = text_input

    else:
        return {"Input Required": 1.0}
    cleaned = clean_resume(raw_text)
    if not cleaned.strip():
        return {"Text Extraction Failed": 1.0}

    vec = tfidf.transform([cleaned])
    probs = model.predict_proba(vec)[0]
    categories = model.classes_

    return {categories[i]: float(probs[i]) for i in range(len(categories))}

In [15]:
def extract_text_from_pdf(pdf_file):
    if pdf_file is None:
        return ""
    with fitz.open(pdf_file.name) as doc:
        text = ""
        for page in doc:
            text += page.get_text()
    return text

def predict_resume(text_input, file_input):
    raw_text = ""

    if file_input is not None:
        try:
            with fitz.open(file_input.name) as doc:
                for page in doc:
                    raw_text += page.get_text()
        except Exception as e:
            return f"Error reading PDF: {str(e)}"

    elif text_input and text_input.strip():
        raw_text = text_input

    else:
        return "Please upload a PDF or paste resume text first!"

    try:
        cleaned = clean_resume(raw_text)

        if not cleaned.strip():
            return "The resume text could not be processed. Is the PDF a scan/image?"

        vec = tfidf.transform([cleaned])
        probs = model.predict_proba(vec)[0]
        categories = model.classes_

        results = {categories[i]: float(probs[i]) for i in range(len(categories))}
        return results
    except Exception as e:
        return f"Prediction Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# AI Resume Category Predictor")
    gr.Markdown("Upload a PDF resume or paste the text to see the predicted job category.")

    with gr.Row():
        with gr.Column():
            text_in = gr.Textbox(label="Paste Resume Text", lines=5)
            file_in = gr.File(label="Or Upload PDF", file_types=[".pdf"])
            submit_btn = gr.Button("Analyze Resume", variant="primary")

        with gr.Column():
            label_out = gr.Label(label="Top Predictions", num_top_classes=3)

    submit_btn.click(fn=predict_resume, inputs=[text_in, file_in], outputs=label_out)

demo.launch(share=True)

/tmp/ipykernel_47521/835389430.py:42: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://773781c93fb9bbce62.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
